# Truy xuat thong tin ket hop KNN

## 1. Xay dung truy van su dung Mo hinh xac suat

In [1]:
import os
import nltk
from nltk import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from whoosh.index import create_in
from whoosh.fields import *
from whoosh.analysis import StandardAnalyzer
from whoosh import qparser
from whoosh import scoring
import whoosh.index as index

import pytrec_eval
import math


nltk.download('punkt_tab')
nltk.download('stopwords')
stoplist = stopwords.words("english")
stoplist.append('oh')
puncts = ['.', ',', ':', '`', '"', "'", '!', '?', "``", "''"]
ps = PorterStemmer()

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
def preprocess(tok, stemmer=ps, punctlist=puncts, stopwords=stoplist):
  tok = tok.lower()
  if tok.isdigit():
    return None
  if tok.isnumeric():
    return None
  if tok in punctlist:
    return None
  if tok in stopwords:
    return None
  return stemmer.stem(tok)

def indexing(src, idx="ind"):
  if src[-1] != '/':
    src += '/'
  schema = Schema(docid=STORED(), content=TEXT(stored=True, analyzer=StandardAnalyzer()))
  ix = create_in(idx, schema)
  writer = ix.writer()

  files = os.listdir(src)
  for f in files:
    r = open(src + f, encoding="cp1252")
    terms = []
    for s in r:
      for sent in sent_tokenize(s.strip()):
        for tok in word_tokenize(sent):
          tok = preprocess(tok)
          if tok != None:
            terms.append(tok)
    r.close()
    cont = " ".join(terms)
    writer.add_document(docid="{}".format(f.split(".")[0]), content=cont)
  writer.commit()

In [3]:
indexing("Cranfield/Cranfield", "ind")

In [4]:
def readGroundTruth(src):
  if src[-1] != '/':
    src += '/'

  GT = {}
  for f in os.listdir(src):
    r = open(src + f)
    rel = {}
    for s in r:
      s = s.strip()
      sp = s.split("\t")
      if len(sp) < 2:
        continue
      did = sp[0].split(" ")[1]
      rel[did] = int(sp[1])
    GT[f.split(".")[0]] = rel
    r.close()
  return GT

In [5]:
GroundTruth = readGroundTruth("Cranfield/RES")
print(GroundTruth)

{'1': {'184': 2, '29': 2, '31': 2, '12': 3, '51': 3, '102': 3, '13': 4, '14': 4, '15': 4, '57': 2, '378': 2, '859': 2, '185': 3, '30': 3, '37': 3, '52': 4, '142': 4, '195': 4, '875': 2, '56': 3, '66': 3, '95': 3, '462': 4, '497': 3, '858': 3, '876': 3, '879': 3, '880': 3, '486': -1}, '10': {'259': 2, '405': 2, '302': 3, '436': 3, '437': 3, '438': 3, '998': 3, '1011': 3, '493': -1}, '100': {'821': 2, '822': 2, '824': 2, '820': 3, '823': 3, '825': 3, '1122': 2, '1051': 3, '1121': 3, '760': -1}, '101': {'817': 2, '818': 2, '819': 2, '820': 3, '825': 3, '824': 2, '760': -1}, '102': {'728': 1, '913': 2, '910': 3, '911': 4, '729': -1}, '103': {'826': 3, '828': 3, '761': -1}, '104': {'833': 3, '834': 3, '835': 3, '836': 3, '837': 3, '762': -1}, '105': {'848': 3, '844': 4, '845': 4, '846': 4, '847': 4, '764': -1}, '106': {'847': 2, '846': 3, '849': 3, '844': 4, '845': 4, '764': -1}, '107': {'725': 2, '728': 2, '729': 3, '911': 3, '720': 4, '75': 4, '909': 4, '640': -1}, '108': {'724': 2, '726'

In [6]:
def readQuery(src):
  qry = {}
  r = open(src)
  for s in r:
    s = s.strip()
    ps = s.split("\t")
    terms = []
    for sent in sent_tokenize(ps[1]):
      for tok in word_tokenize(sent):
        tok = preprocess(tok)
        if tok != None:
          terms.append(tok)
    qry[ps[0]] = " ".join(terms)
  return qry

In [7]:
Queries = readQuery("Cranfield/query.txt")
print(Queries)

{'1': 'similar law must obey construct aeroelast model heat high speed aircraft', '2': 'structur aeroelast problem associ flight high speed aircraft', '3': 'problem heat conduct composit slab solv far', '4': 'criterion develop show empir valid flow solut chemic react ga mixtur base simplifi assumpt instantan local chemic equilibrium', '5': 'chemic kinet system applic hyperson aerodynam problem', '6': 'theoret experiment guid turbul couett flow behaviour', '7': 'possibl relat avail pressur distribut ogiv forebodi zero angl attack lower surfac pressur equival ogiv forebodi angl attack', '8': 'method -dash exact approxim -dash present avail predict bodi pressur angl attack', '9': 'paper intern /slip flow/ heat transfer studi', '10': 'real-ga transport properti air avail wide rang enthalpi densiti', '11': 'possibl find analyt similar solut strong blast wave problem newtonian approxim', '12': 'aerodynam perform channel flow ground effect machin calcul', '13': 'basic mechan transon aileron b

In [8]:
def processQueries(ind, qry):

  idx = index.open_dir(ind)
  searcher = idx.searcher(weighting=scoring.BM25F())
  parser = qparser.QueryParser("content", idx.schema, group=qparser.OrGroup)

  RET = {}
  for key in qry:
    query = parser.parse(qry[key])
    results = searcher.search(query, limit=None)
    rel = {}
    for i in range(len(results)):
      rel[results[i]["docid"]] = results[i].score
    RET[key] = rel
  return RET

In [9]:
RunResults = processQueries("ind", Queries)

In [10]:
print(pytrec_eval.supported_measures)
eval = pytrec_eval.RelevanceEvaluator(GroundTruth, ["infAP", "11pt_avg"])
Results = eval.evaluate(RunResults)

MAP = 0
MAP11PT = 0

for key in Results:
  value = Results[key]["infAP"]
  if not math.isnan(value):
    MAP += value
  value = Results[key]["11pt_avg"]
  if not math.isnan(value):
    MAP11PT += value

MAP /= len(Results)
MAP11PT /= len(Results)

print(MAP, MAP11PT)

{'num_ret', 'set_F', 'ndcg', 'relstring', 'infAP', 'Rprec', 'map_cut', 'recall', 'set_relative_P', 'P', 'set_map', 'iprec_at_recall', 'relative_P', 'num_q', 'bpref', 'success', 'num_nonrel_judged_ret', 'Rprec_mult', 'gm_bpref', 'G', 'runid', 'num_rel', 'set_recall', 'set_P', 'num_rel_ret', 'utility', 'ndcg_cut', 'ndcg_rel', 'gm_map', 'binG', 'map', '11pt_avg', 'recip_rank', 'Rndcg'}
0.35099879106595755 0.3218448003963703


## 2. Mo hinh KNN

### 1. Chuan bi du lieu KNN (TF-IDF)

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

Lay toan bo tai lieu da tien xu ly

In [12]:
def load_docs_for_knn(src):
    docs = {}
    for f in os.listdir(src):
        r = open(os.path.join(src, f), encoding="cp1252")
        terms = []
        for s in r:
            for sent in sent_tokenize(s.strip()):
                for tok in word_tokenize(sent):
                    tok = preprocess(tok)
                    if tok:
                        terms.append(tok)
        r.close()
        docs[f.split(".")[0]] = " ".join(terms)
    return docs

Docs = load_docs_for_knn("Cranfield/Cranfield")


### 2. Xay dung mo hinh KNN (TF-IDF + Cosine)

In [13]:
doc_ids = list(Docs.keys())
doc_texts = list(Docs.values())

vectorizer = TfidfVectorizer()
X_docs = vectorizer.fit_transform(doc_texts)


### 3. Truy van bang KNN

In [14]:
def processQueries_KNN(queries, topk=1000):
    RET = {}

    for qid, qtext in queries.items():
        q_vec = vectorizer.transform([qtext])
        sims = cosine_similarity(q_vec, X_docs)[0]

        ranked = sorted(
            zip(doc_ids, sims),
            key=lambda x: x[1],
            reverse=True
        )

        rel = {}
        for docid, score in ranked[:topk]:
            if score > 0:
                rel[docid] = float(score)

        RET[qid] = rel
    return RET

RunResults_KNN = processQueries_KNN(Queries)

## Ket hop 2 mo hinh de truy van

Chuẩn hoá score (min-max)

In [15]:
def normalize_scores(run):
    norm_run = {}
    for qid, docs in run.items():
        if not docs:
            norm_run[qid] = {}
            continue
        scores = list(docs.values())
        min_s, max_s = min(scores), max(scores)
        norm_run[qid] = {}
        for d, s in docs.items():
            if max_s > min_s:
                norm_run[qid][d] = (s - min_s) / (max_s - min_s)
            else:
                norm_run[qid][d] = 0.0
    return norm_run


Fusion

In [16]:
def fuse_runs(bm25, knn, alpha=0.6):
    bm25 = normalize_scores(bm25)
    knn = normalize_scores(knn)

    FUSED = {}

    for qid in bm25:
        docs = set(bm25[qid].keys()) | set(knn.get(qid, {}).keys())
        fused_scores = {}
        for d in docs:
            fused_scores[d] = (
                alpha * bm25[qid].get(d, 0) +
                (1 - alpha) * knn.get(qid, {}).get(d, 0)
            )
        FUSED[qid] = fused_scores

    return FUSED


In [17]:
RunResults_Fused = fuse_runs(RunResults, RunResults_KNN, alpha=0.6)


## Danh gia

### Truy van ban dau

In [18]:
print(pytrec_eval.supported_measures)
eval = pytrec_eval.RelevanceEvaluator(GroundTruth, ["infAP", "11pt_avg"])
Results = eval.evaluate(RunResults)

MAP = 0
MAP11PT = 0

for key in Results:
  value = Results[key]["infAP"]
  if not math.isnan(value):
    MAP += value
  value = Results[key]["11pt_avg"]
  if not math.isnan(value):
    MAP11PT += value

MAP /= len(Results)
MAP11PT /= len(Results)

print(MAP, MAP11PT)

{'num_ret', 'set_F', 'ndcg', 'relstring', 'infAP', 'Rprec', 'map_cut', 'recall', 'set_relative_P', 'P', 'set_map', 'iprec_at_recall', 'relative_P', 'num_q', 'bpref', 'success', 'num_nonrel_judged_ret', 'Rprec_mult', 'gm_bpref', 'G', 'runid', 'num_rel', 'set_recall', 'set_P', 'num_rel_ret', 'utility', 'ndcg_cut', 'ndcg_rel', 'gm_map', 'binG', 'map', '11pt_avg', 'recip_rank', 'Rndcg'}
0.35099879106595755 0.3218448003963703


In [19]:
def average_precision_at_k(ranked_docs, relevant_docs, k=20):
    score = 0.0
    hits = 0
    for i, doc_id in enumerate(ranked_docs[:k], start=1):
        if doc_id in relevant_docs:
            hits += 1
            score += hits / i
    if not relevant_docs:
        return 0.0
    return score / min(len(relevant_docs), k)


MAP20 = 0
for qid in RunResults:
    ranked_docs = list(RunResults[qid].keys())
    relevant_docs = {
        doc_id for doc_id, rel in GroundTruth[qid].items() if rel > 0
    }
    MAP20 += average_precision_at_k(ranked_docs, relevant_docs, k=20)

MAP20 /= len(RunResults)
print("MAP@20 =", MAP20)


MAP@20 = 0.26671828775677286


### Ket hop KNN

In [24]:
Results_Fused = eval.evaluate(RunResults_Fused)

In [25]:
import math
import pytrec_eval

metrics = [
    "map",
    "infAP",
    "11pt_avg",
    "P_5", "P_10", "P_20",
    "recall_5", "recall_10", "recall_20"
]

evaluator = pytrec_eval.RelevanceEvaluator(GroundTruth, metrics)
results = evaluator.evaluate(RunResults_Fused)

# Khởi tạo
MAP = infAP = MAP11PT = 0.0
P5 = P10 = P20 = 0.0
R5 = R10 = R20 = 0.0

valid_queries = 0

for qid, res in results.items():
    if not math.isnan(res["map"]):
        valid_queries += 1

        MAP += res["map"]
        infAP += res["infAP"]
        MAP11PT += res["11pt_avg"]

        P5 += res["P_5"]
        P10 += res["P_10"]
        P20 += res["P_20"]

        R5 += res["recall_5"]
        R10 += res["recall_10"]
        R20 += res["recall_20"]

# Trung bình
MAP /= valid_queries
infAP /= valid_queries
MAP11PT /= valid_queries

P5 /= valid_queries
P10 /= valid_queries
P20 /= valid_queries

R5 /= valid_queries
R10 /= valid_queries
R20 /= valid_queries

# 🔹 TÍNH F1@k
def f1(p, r):
    return 0.0 if (p + r) == 0 else 2 * p * r / (p + r)

F1_5 = f1(P5, R5)
F1_10 = f1(P10, R10)
F1_20 = f1(P20, R20)


In [26]:
print("=== BM25 + KNN Fusion ===")
print(f"Queries evaluated : {valid_queries}")

print(f"MAP        : {MAP:.4f}")
print(f"infAP      : {infAP:.4f}")
print(f"11pt Avg   : {MAP11PT:.4f}")

print(f"P@5        : {P5:.4f}")
print(f"R@5        : {R5:.4f}")
print(f"F1@5       : {F1_5:.4f}")

print(f"P@10       : {P10:.4f}")
print(f"R@10       : {R10:.4f}")
print(f"F1@10      : {F1_10:.4f}")

print(f"P@20       : {P20:.4f}")
print(f"R@20       : {R20:.4f}")
print(f"F1@20      : {F1_20:.4f}")


=== BM25 + KNN Fusion ===
Queries evaluated : 225
MAP        : 0.3107
infAP      : 0.3653
11pt Avg   : 0.3346
P@5        : 0.3164
R@5        : 0.2849
F1@5       : 0.2999
P@10       : 0.2378
R@10       : 0.3962
F1@10      : 0.2972
P@20       : 0.1640
R@20       : 0.5206
F1@20      : 0.2494


In [22]:
# Tính trung bình (macro average) cho tất cả query
avg_metrics = {}
for m in next(iter(results.values())).keys():
    avg_metrics[m] = sum(q[m] for q in results.values()) / len(results)

# Tính macro F1 từ Precision và Recall trung bình
cutoffs = [5, 10, 20]

# Tính macro F1 cho các cutoff
for k in cutoffs:
    p_key = f"P_{k}"
    r_key = f"recall_{k}"
    f1_key = f"F1_{k}"
    
    if p_key in avg_metrics and r_key in avg_metrics:
        p = avg_metrics[p_key]
        r = avg_metrics[r_key]
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
        avg_metrics[f1_key] = f1

# In ra
print("Average over all queries:")
for m, v in avg_metrics.items():
    print(f"  {m:10s}: {v:.4f}")

Average over all queries:
  map       : 0.2974
  P_5       : 0.2969
  P_10      : 0.2302
  P_20      : 0.1584
  recall_5  : 0.2757
  recall_10 : 0.3857
  recall_20 : 0.5029
  infAP     : 0.3510
  11pt_avg  : 0.3218
  F1_5      : 0.2859
  F1_10     : 0.2883
  F1_20     : 0.2410


### So sanh & Ket luan
